# ETF Fragility Through Market Cycles

Motivating figure for the paper: how does the `vol_12w` fragility signal evolve for representative fixed-income ETFs through key stress episodes (COVID 2020, rate hike cycle 2022)?

Representative ETFs span the fragility spectrum:
- **TLT** — iShares 20+ Year Treasury (long duration, high rate sensitivity)
- **HYG** — iShares High Yield Corporate Bond (credit risk)
- **LQD** — iShares IG Corporate Bond (investment grade)
- **BND** — Vanguard Total Bond Market (broad aggregate)
- **EMB** — iShares EM USD Bond (emerging markets)
- **AGG** — iShares Core US Aggregate (benchmark)

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from src import config
from src.features.rolling_risk import add_rolling_risk_metrics
from src.features.stress_index import add_stress_index

plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 9,
})

panel = pd.read_csv(config.CORE_PANEL_CSV, parse_dates=['Date', 'Inception'])
panel = panel.sort_values(['Symbol', 'Date']).reset_index(drop=True)

if 'vol_12w' not in panel.columns:
    panel = add_rolling_risk_metrics(panel)
if 'stress_index' not in panel.columns:
    panel = add_stress_index(panel)

print(f'Panel: {len(panel):,} rows  |  {panel.Symbol.nunique()} ETFs  |  '
      f'{panel.Date.min().date()} to {panel.Date.max().date()}')

ETFS = ['TLT', 'HYG', 'LQD', 'BND', 'EMB', 'AGG']
available = [t for t in ETFS if t in panel['Symbol'].values]
missing   = [t for t in ETFS if t not in panel['Symbol'].values]
print(f'Representative ETFs found: {available}')
if missing:
    print(f'Not in universe: {missing}')

COLORS = {
    'TLT': '#1f77b4',   # blue   — long Treasury
    'HYG': '#d62728',   # red    — high yield
    'LQD': '#ff7f0e',   # orange — IG corp
    'BND': '#2ca02c',   # green  — total market
    'EMB': '#9467bd',   # purple — EM
    'AGG': '#8c564b',   # brown  — aggregate
}

In [ ]:
# ── Fragility time series — vol_12w for representative ETFs ───────────────
rep = panel[panel['Symbol'].isin(available)].dropna(subset=['vol_12w']).copy()

stress_ts = (panel[['Date', 'high_stress']]
             .drop_duplicates('Date').set_index('Date').sort_index())

fig, ax = plt.subplots(figsize=(11, 4.5))

# Stress shading
ax.fill_between(stress_ts.index, 0, 1,
                where=stress_ts['high_stress'] == 1,
                transform=ax.get_xaxis_transform(),
                alpha=0.10, color='firebrick', label='High-stress weeks')

for sym in available:
    s = rep[rep['Symbol'] == sym].set_index('Date')['vol_12w']
    ax.plot(s.index, s * 100, label=sym, color=COLORS.get(sym), lw=1.3)

ax.set_ylabel('12-week rolling volatility (%)')
ax.set_title('Fixed-income ETF fragility (vol_12w) through market cycles\n'
             'Red shading = composite macro stress index > 1σ')
ax.legend(ncol=4, fontsize=8, loc='upper left')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
plt.tight_layout()
plt.show()

# Summary stats per ETF
summary = (rep.groupby('Symbol')['vol_12w']
           .agg(mean='mean', std='std',
                p10=lambda x: x.quantile(0.1),
                p90=lambda x: x.quantile(0.9),
                max='max')
           .loc[available]
           .round(5))
summary.columns = ['Mean', 'Std', 'P10', 'P90', 'Max']
print('vol_12w summary by ETF:')
display(summary)

In [ ]:
# ── Cross-sectional fragility distribution over time ───────────────────────
# Median + IQR band across all ETFs: shows how the typical fragility level
# and its dispersion shift through market regimes.

univ = panel[panel['category_bucket'] != 'Other'].dropna(subset=['vol_12w'])

weekly_pct = (univ.groupby('Date')['vol_12w']
              .quantile([0.25, 0.50, 0.75])
              .unstack()
              .rename(columns={0.25: 'q25', 0.50: 'med', 0.75: 'q75'})
              .sort_index())

fig, ax = plt.subplots(figsize=(11, 4))

ax.fill_between(weekly_pct.index, 0, 1,
                where=stress_ts.reindex(weekly_pct.index)['high_stress'] == 1,
                transform=ax.get_xaxis_transform(),
                alpha=0.10, color='firebrick', label='High-stress weeks')

ax.fill_between(weekly_pct.index,
                weekly_pct['q25'] * 100, weekly_pct['q75'] * 100,
                alpha=0.25, color='steelblue', label='IQR (Q1–Q3)')
ax.plot(weekly_pct.index, weekly_pct['med'] * 100,
        color='steelblue', lw=1.4, label='Median vol_12w')

# Overlay representative ETFs
for sym in ['TLT', 'HYG']:
    if sym in available:
        s = rep[rep['Symbol'] == sym].set_index('Date')['vol_12w']
        ax.plot(s.index, s * 100, color=COLORS[sym], lw=1.0,
                ls='--', label=sym, alpha=0.8)

ax.set_ylabel('12-week rolling volatility (%)')
ax.set_title('Cross-sectional fragility distribution over time\n'
             'Band = IQR across all non-Other ETFs; dashed = TLT, HYG')
ax.legend(ncol=3, fontsize=8)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
plt.tight_layout()
plt.show()

In [ ]:
# ── Fragility rankings during key stress episodes ─────────────────────────
# Snapshot vol_12w around COVID peak (Mar 2020) and rate-hike peak (Oct 2022)
# to show which ETF types were most fragile during each episode.

EPISODES = {
    'COVID peak\n(Mar 2020)':     ('2020-02-01', '2020-04-30'),
    'Rate-hike cycle\n(2022)':    ('2022-01-01', '2022-12-31'),
}

ep_rows = []
for label, (start, end) in EPISODES.items():
    window = panel[(panel['Date'] >= start) & (panel['Date'] <= end)]
    if window.empty:
        continue
    avg = (window.groupby('Symbol')['vol_12w'].mean()
           .dropna()
           .sort_values(ascending=False))
    ep_rows.append({'episode': label, 'ranking': avg})

if not ep_rows:
    print('No data for the specified stress episodes — check panel date range.')
else:
    n_ep = len(ep_rows)
    fig, axes = plt.subplots(1, n_ep, figsize=(6 * n_ep, 4.5), sharey=False)
    if n_ep == 1:
        axes = [axes]

    for ax, row in zip(axes, ep_rows):
        top = row['ranking'].head(15)
        colors = [COLORS.get(s, '#aec7e8') for s in top.index]
        bars = ax.barh(range(len(top)), top.values * 100,
                       color=colors, edgecolor='white', linewidth=0.5)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top.index, fontsize=8)
        ax.invert_yaxis()
        ax.set_xlabel('Avg vol_12w (%)')
        ax.set_title(f'Top-15 fragile ETFs\n{row["episode"]}', fontsize=9)
        ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))

        # Highlight representative ETFs
        for i, sym in enumerate(top.index):
            if sym in COLORS:
                ax.get_yticklabels()[i].set_fontweight('bold')

    plt.suptitle('Fragility rankings during key stress episodes\n'
                 'Bold labels = representative ETFs from cell above', fontsize=10, y=1.02)
    plt.tight_layout()
    plt.show()

    # Print the rep-ETF ranks within each episode
    print('\nRepresentative ETF percentile rank during each episode (lower = more fragile):')
    for row in ep_rows:
        n = len(row['ranking'])
        ranks = {s: round(100 * (row['ranking'].index.get_loc(s) + 1) / n, 1)
                 for s in available if s in row['ranking'].index}
        ep_label = row['episode'].replace('\n', ' ')
        print(f"  {ep_label}: " + '  |  '.join(f"{s}: {r:.0f}th pct" for s, r in ranks.items()))

In [ ]:
# ── Time-in-top-quartile: which ETFs are persistently fragile? ─────────────
# For each ETF, compute the fraction of weeks it spent in the top 25%
# of the cross-sectional vol_12w distribution.
# High fractions → structural fragility, not just episode-specific spikes.

# Cross-sectional Q3 cutoff each week
weekly_q75 = (panel.dropna(subset=['vol_12w'])
              .groupby('Date')['vol_12w']
              .quantile(0.75)
              .rename('q75'))

frag = (panel.dropna(subset=['vol_12w'])
        .join(weekly_q75, on='Date')
        .assign(top_q=lambda d: (d['vol_12w'] >= d['q75']).astype(int)))

pct_top = (frag.groupby('Symbol')
           .agg(pct_top_q=('top_q', 'mean'),
                n_weeks=('top_q', 'count'))
           .query('n_weeks >= 52')          # require at least 1 year of data
           .sort_values('pct_top_q', ascending=False))

# ── Bar chart: top 20 persistently fragile ETFs ───────────────────────────
top20 = pct_top.head(20)
rep_in_top20 = [s for s in available if s in top20.index]

fig, ax = plt.subplots(figsize=(10, 4.5))
bar_colors = [COLORS.get(s, '#aec7e8') for s in top20.index]
ax.bar(range(len(top20)), top20['pct_top_q'] * 100,
       color=bar_colors, edgecolor='white', linewidth=0.5)
ax.axhline(25, color='black', lw=0.8, ls='--', label='25% baseline')
ax.set_xticks(range(len(top20)))
ax.set_xticklabels(top20.index, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('% of weeks in top fragility quartile')
ax.set_title('Persistently fragile ETFs — time spent in top vol_12w quartile\n'
             'Colored bars = representative ETFs; dashed = 25% random baseline')
ax.legend(fontsize=8)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
plt.tight_layout()
plt.show()

# ── Representative ETF comparison table ───────────────────────────────────
if available:
    rep_tiq = pct_top.loc[[s for s in available if s in pct_top.index]].copy()
    rep_tiq['pct_top_q_pct'] = (rep_tiq['pct_top_q'] * 100).round(1)
    rep_tiq['rank'] = rep_tiq['pct_top_q'].rank(ascending=False, method='min').astype(int)
    rep_tiq = rep_tiq[['pct_top_q_pct', 'n_weeks', 'rank']].rename(
        columns={'pct_top_q_pct': '% wks top-Q', 'n_weeks': 'N weeks', 'rank': 'Rank (universe)'})
    print('Time-in-top-quartile for representative ETFs:')
    display(rep_tiq)

print(f'\nUniverse: {len(pct_top)} ETFs with ≥52 weeks of vol_12w data')
print(f'Overall mean % top-Q: {pct_top["pct_top_q"].mean()*100:.1f}%  '
      f'(expected ~25% if uniform)')